In [2]:
"""
Random Forest CV — 12 cities (Houston + Phoenix + 10 new)
  - Per-city 5-fold CV: baseline / embed-only / baseline+embed
  - LOCO CV (embed-only) for cross-city generalisation

Run from rq1_explainability/ root:
  jupyter notebook notebooks/02_random_forest_cv.ipynb

Note: poverty_rate / high_poverty_flag are only available for Houston + Phoenix.
      For the 10 new cities, demo features are filled with 0.
      embed-only results are valid for all 12 cities.
"""

import pandas as pd, numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

ROOT = Path("..").resolve()  # notebooks/ is one level below project root
df   = pd.read_csv(ROOT / "outputs/modeling_table.csv", dtype={"GEOID": str})

EMBED_COLS = [f"A{i:02d}" for i in range(64)]
DEMO_COLS  = ["poverty_rate", "high_poverty_flag"]   # available all cities (filled 0 where missing)
CITIES     = sorted(df["city"].unique().tolist())

# ── Target: gap_z = z(HI) - z(LST), per city ─────────────────────────────────
def compute_gap_z(grp):
    def zs(s): return (s - s.mean()) / s.std() if s.std() > 0 else s * 0
    return zs(grp["hi_c"]) - zs(grp["lst_c"])

df["gap_z"] = df.groupby("city", group_keys=False).apply(
    compute_gap_z, include_groups=False
)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 68)
print("Per-City 5-Fold CV  (baseline / embed-only / baseline+embed)")
print("=" * 68)

results = []
for city in CITIES:
    sub = df[df["city"] == city].dropna(subset=EMBED_COLS + ["gap_z"]).copy()
    sub[DEMO_COLS] = sub[DEMO_COLS].fillna(0)   # fill missing demo with 0
    y = sub["gap_z"]
    has_demo = df[df["city"] == city]["poverty_rate"].notna().any()

    if len(sub) < 20:
        print(f"  {city:15s} | skipped ({len(sub)} rows)")
        continue

    feature_sets = {
        "baseline":       sub[DEMO_COLS],
        "embed-only":     sub[EMBED_COLS],
        "baseline+embed": sub[DEMO_COLS + EMBED_COLS],
    }
    for label, X in feature_sets.items():
        scores  = cross_val_score(rf, X, y, cv=cv, scoring="r2")
        r2_mean = np.mean(scores)
        r2_std  = np.std(scores)
        tag = "" if has_demo else " *"
        print(f"  {city:15s} | {label:16s} | R²={r2_mean:+.3f} (±{r2_std:.3f})  n={len(sub)}{tag}")
        results.append({"city": city, "features": label,
                        "r2_mean": round(r2_mean, 4), "r2_std": round(r2_std, 4),
                        "n": len(sub), "has_acs_demo": has_demo})

print("\n* = demo features filled with 0 (no ACS data)")

# ── LOCO CV ───────────────────────────────────────────────────────────────────
print()
print("=" * 68)
print("LOCO CV  (embed-only, leave-one-city-out)")
print("=" * 68)

valid    = df[EMBED_COLS + ["gap_z"]].notna().all(axis=1)
X_embed  = df.loc[valid, EMBED_COLS]
y_all    = df.loc[valid, "gap_z"]
city_col = df.loc[valid, "city"]

loco_rows = []
for test_city in CITIES:
    tr = city_col != test_city
    te = city_col == test_city
    if te.sum() < 10: continue
    rf.fit(X_embed[tr], y_all[tr])
    r2 = rf.score(X_embed[te], y_all[te])
    print(f"  test={test_city:15s}  R²={r2:+.3f}  (n_test={te.sum()})")
    loco_rows.append({"test_city": test_city, "r2_loco": round(r2, 4), "n_test": int(te.sum())})

# ── Save ──────────────────────────────────────────────────────────────────────
(ROOT / "outputs").mkdir(exist_ok=True)
pd.DataFrame(results).to_csv(ROOT / "outputs/model_comparison.csv", index=False)
pd.DataFrame(loco_rows).to_csv(ROOT / "outputs/loco_cv_results.csv", index=False)
print(f"\nSaved → outputs/model_comparison.csv")
print(f"Saved → outputs/loco_cv_results.csv")


Per-City 5-Fold CV  (baseline / embed-only / baseline+embed)
  atlanta         | baseline         | R²=-0.024 (±0.013)  n=223 *
  atlanta         | embed-only       | R²=+0.782 (±0.024)  n=223 *
  atlanta         | baseline+embed   | R²=+0.783 (±0.024)  n=223 *
  boston          | baseline         | R²=-0.071 (±0.063)  n=191 *
  boston          | embed-only       | R²=+0.355 (±0.227)  n=191 *
  boston          | baseline+embed   | R²=+0.358 (±0.225)  n=191 *
  chicago         | baseline         | R²=-0.003 (±0.004)  n=872 *
  chicago         | embed-only       | R²=+0.504 (±0.040)  n=872 *
  chicago         | baseline+embed   | R²=+0.504 (±0.042)  n=872 *
  dallas          | baseline         | R²=-0.013 (±0.011)  n=449 *
  dallas          | embed-only       | R²=+0.535 (±0.069)  n=449 *
  dallas          | baseline+embed   | R²=+0.537 (±0.069)  n=449 *
  houston         | baseline         | R²=-0.325 (±0.123)  n=549
  houston         | embed-only       | R²=+0.561 (±0.086)  n=549
  hou